# HS4002 Week 11

## Panel Data Models

Panel data follows the **same units** (people, players, countries) across multiple time points, allowing us to control for unobserved unit-level heterogeneity.

In Python, the `linearmodels` library provides panel regression — equivalent to R's `plm` package.

We'll use NBA player data (2013–2017) to ask: **how does a player's age affect their scoring performance?**

In [2]:
%pip install linearmodels stargazer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [linearmodels] [linearmodels]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# !pip install pandas numpy plotnine linearmodels stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS

## Setup: Load and Prepare the NBA Dataset

In [4]:
# Data source: https://www.kaggle.com/datasets/drgilermo/nba-players-stats
# NBA player statistics 1950-2017 from basketball-reference.com
raw_df = pd.read_csv("bball.csv").iloc[:, 1:]  # drop the unnamed index column

df = (
	raw_df[raw_df["Year"] >= 2013]
	.groupby(["Player", "Year"])
	.agg(
		games_played    = ("G",    "sum"),
		games_started   = ("GS",   "sum"),
		minutes_played  = ("MP",   "sum"),
		age             = ("Age",  "mean"),
		personal_fouls  = ("PF",   "sum"),
		points          = ("PTS",  "sum"),
		true_shooting   = ("TS%",  "mean"),
		assists         = ("AST",  "sum"),
		steals          = ("STL",  "sum"),
		blocks          = ("BLK",  "sum"),
		fga             = ("FGA",  "sum"),
		fg              = ("FG",   "sum"),
		threeP          = ("3P",   "sum"),
		threePA         = ("3PA",  "sum"),
		twoP            = ("2P",   "sum"),
		twoPA           = ("2PA",  "sum"),
		ft              = ("FT",   "sum"),
		fta             = ("FTA",  "sum"),
		def_rb          = ("DRB",  "sum"),
		off_rb          = ("ORB",  "sum"),
	)
	.assign(ppg=lambda x: x["points"] / x["games_played"])
	.reset_index()
)
print(df.shape)
df.head()

(2403, 23)


,Player,Year,games_played,games_started,minutes_played,age,personal_fouls,points,true_shooting,assists,...,fg,threeP,threePA,twoP,twoPA,ft,fta,def_rb,off_rb,ppg
0,A.J. Hammons,2017.0,22.0,0.0,163.0,24.0,21.0,48.0,0.472000,4.0,...,17.0,5.0,10.0,12.0,32.0,9.0,20.0,28.0,8.0,2.181818
1,A.J. Price,2013.0,57.0,22.0,1278.0,26.0,73.0,441.0,0.501000,205.0,...,161.0,70.0,200.0,91.0,213.0,49.0,62.0,94.0,20.0,7.736842
2,A.J. Price,2014.0,28.0,0.0,99.0,27.0,5.0,44.0,0.469000,13.0,...,19.0,6.0,22.0,13.0,24.0,0.0,2.0,9.0,1.0,1.571429
3,A.J. Price,2015.0,52.0,0.0,648.0,28.0,30.0,266.0,0.376750,92.0,...,102.0,30.0,114.0,72.0,160.0,32.0,48.0,52.0,12.0,5.115385
4,Aaron Brooks,2013.0,106.0,40.0,1994.0,28.0,188.0,752.0,0.500667,228.0,...,286.0,100.0,268.0,186.0,364.0,80.0,104.0,134.0,26.0,7.094340


## Setting up the Panel Index

`linearmodels` requires a **MultiIndex** of (entity, time). Equivalent to the `index=c('Player','Year')` argument in R's `plm()`.

In [5]:
# Set the panel index — required by linearmodels
df_panel = df.set_index(['Player', 'Year'])

Note: Have to specify to computer what is time column and entity column

# A Tutorial for Panel Models

## Pooled OLS

Equivalent to a regular `lm()` in R — ignores the panel structure.

In [6]:
pool_model = PooledOLS.from_formula('ppg ~ age + I(age**2)', data=df_panel)
pool_result = pool_model.fit()
print(pool_result.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                    ppg   R-squared:                        0.6757
Estimator:                  PooledOLS   R-squared (Between):              0.6103
No. Observations:                2403   R-squared (Within):               0.0318
Date:                Wed, Oct 28 2026   R-squared (Overall):              0.6757
Time:                        12:07:32   Log-likelihood                   -7587.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2501.2
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             2501.2
                            

We can observed a quadratic effect between points per game (ppg) and age. This means ppg initially increases with age but after it reach a turning point, ppg start to decrease as age increase.

## Fixed Effects (Within) Model

Adds `EntityEffects` to control for time-invariant player characteristics. Equivalent to `plm(..., model='within')` in R.

In [7]:
fe_model = PanelOLS.from_formula('ppg ~ age + I(age**2) + EntityEffects', data=df_panel)
fe_result = fe_model.fit()
print(fe_result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    ppg   R-squared:                        0.1774
Estimator:                   PanelOLS   R-squared (Between):             -66.812
No. Observations:                2403   R-squared (Within):               0.1774
Date:                Wed, Oct 28 2026   R-squared (Overall):             -43.473
Time:                        12:12:33   Log-likelihood                   -5167.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      170.80
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,1584)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             170.80
                            

Interpret the coefficients. How does the fixed-effects estimate differ from the pooled OLS estimate?

Interpretation: 

## Random Effects Model

Equivalent to `plm(..., model='random')` in R.

In [8]:
re_model = RandomEffects.from_formula('ppg ~ age + I(age**2)', data=df_panel)
re_result = re_model.fit()
print(re_result.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                    ppg   R-squared:                        0.3804
Estimator:              RandomEffects   R-squared (Between):              0.6371
No. Observations:                2403   R-squared (Within):               0.0412
Date:                Wed, Oct 28 2026   R-squared (Overall):              0.6573
Time:                        12:12:44   Log-likelihood                   -5810.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      737.02
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             737.02
                            

Procude result much closer to OLS model: RE takes in a lot of between in variables (take factors from other players)

## Model Diagnostics

### Lagrange Multiplier Test

Tests whether individual effects are present (i.e., whether FE/RE is needed over pooled OLS). Equivalent to R's `plmtest(fe_model, effect='individual')`.

In [9]:
# Breusch-Pagan LM test for random effects
# Under H0 (no individual effects), we prefer pooled OLS.
# Equivalent to R's plmtest(..., effect='individual', type='bp').
from scipy.stats import chi2

resid = pool_result.resids
e_i   = resid.groupby(level=0).sum()
ssr   = float((resid ** 2).sum())
A1    = float((e_i ** 2).sum()) / ssr - 1
T_i   = resid.groupby(level=0).size()
n_obs = resid.shape[0]
lm_stat = (n_obs ** 2) / (2 * (float((T_i ** 2).sum()) - n_obs)) * A1 ** 2
lm_pval = chi2.sf(lm_stat, df=1)

print('LM test for individual effects:')
print(f'LM statistic = {lm_stat:.4f}')
print('df = 1')
print(f'p-value = {lm_pval:.4f}')

LM test for individual effects:
LM statistic = 2381.2182
df = 1
p-value = 0.0000


As p-value is very small, we reject null hypothesis. This means that there is significant variation across players that did not capture by the included regressors, thus it is better to use random effect model rather than pooled OLS model, as RE model captures the omitted variables. 

### Hausman Test

Tests whether RE estimates are consistent (i.e., whether FE or RE is preferred). Under H0, RE is consistent and more efficient. Equivalent to R's `phtest(fe_model, re_model)`.

In [10]:
# Hausman test: compare FE and RE estimates
# Under H0 (RE consistent), the difference in coefficients should be small
from scipy.stats import chi2

common_params = fe_result.params.index.intersection(re_result.params.index)

beta_diff = fe_result.params[common_params] - re_result.params[common_params]
cov_diff  = (
	fe_result.cov.loc[common_params, common_params] -
	re_result.cov.loc[common_params, common_params]
)

h_stat = float(beta_diff.T @ np.linalg.inv(cov_diff.values) @ beta_diff)
p_val  = chi2.sf(h_stat, df=len(common_params))
print(f'Hausman statistic = {h_stat:.4f}')
print(f'df = {len(common_params)}')
print(f'p-value = {p_val:.4f}')
print('Prefer FE over RE:', p_val < 0.05)

Hausman statistic = 285.0094
df = 2
p-value = 0.0000
Prefer FE over RE: True


The null hypothesis is that the time-invariant individual are not correlated with the regressors. However, as p<0.001, we reject null and infer that the time-invariant individual are correlated with the regressors. Thus, we want to use fixed effect model, rather than random effect model, as random effect model assume that the unobserved time-invariant factors are uncorrelated with each explanatory variable.

## Comparing the Models

As in earlier weeks, `stargazer` collects the fitted models into one regression table. Reading pooled OLS, fixed effects, and random effects side by side is easier than scrolling through three summaries. Equivalent to R's `stargazer(lm1, fe_glm1, re_glm1)`.

In [11]:
from stargazer.stargazer import Stargazer

sg = Stargazer([pool_result, fe_result, re_result])
sg.title('Points per Game and Age')

# Label the models and variables readably, rather than by raw column name
sg.dependent_variable_name('Points per game')
sg.custom_columns(['Pooled OLS', 'Fixed Effects', 'Random Effects'], [1, 1, 1])
sg.covariate_order(['age', 'I(age ** 2)'])
sg.rename_covariates({
	'age':         'Age',
	'I(age ** 2)': 'Age-squared',
})

# Ending the cell on the object itself renders the table in the notebook
sg

This table alone cannot determine which model is the best among the three models. Need further Lagrange Multiplier (LM) Diagnostics or Hausman Test. 

# Independent Analysis: Has Sports Science Extended Career Peaks?

Compare players from 2008–2012 with players from 2013–2017. Does age affect scoring differently across the two eras?

# (1) create two data sets, one for each era.

In [19]:
# (1) create two data sets, one for each era.
# Create a separate dataset for 2008–2017
df_2008_2017 = (
    raw_df[raw_df["Year"].between(2008, 2017)]
    .groupby(["Player", "Year"])
    .agg(
        games_played   = ("G", "sum"),
        games_started  = ("GS", "sum"),
        minutes_played = ("MP", "sum"),
        age            = ("Age", "mean"),
        personal_fouls = ("PF", "sum"),
        points         = ("PTS", "sum"),
        true_shooting  = ("TS%", "mean"),
        assists        = ("AST", "sum"),
        steals         = ("STL", "sum"),
        blocks         = ("BLK", "sum"),
        fga            = ("FGA", "sum"),
        fg             = ("FG", "sum"),
        threeP         = ("3P", "sum"),
        threePA        = ("3PA", "sum"),
        twoP           = ("2P", "sum"),
        twoPA          = ("2PA", "sum"),
        ft             = ("FT", "sum"),
        fta            = ("FTA", "sum"),
        def_rb         = ("DRB", "sum"),
        off_rb         = ("ORB", "sum"),
    )
    .assign(ppg=lambda x: x["points"] / x["games_played"])
    .reset_index()
)

In [20]:
# Create two datasets
df_2008_2012 = df_2008_2017[df_2008_2017["Year"].between(2008, 2012)].copy()
df_2013_2017 = df_2008_2017[df_2008_2017["Year"].between(2013, 2017)].copy()


In [21]:
# Set the panel index
panel_2008_2012 = df_2008_2012.set_index(["Player", "Year"])
panel_2013_2017 = df_2013_2017.set_index(["Player", "Year"])

In [22]:
# Check the datasets
print("2008_2012:", panel_2008_2012.shape)

2008_2012: (2266, 21)


In [23]:
# Check the datasets
print("2013_2017:", panel_2013_2017.shape)

2013_2017: (2403, 21)


# # (2) fit models for each of the two data sets

In [25]:
# (2) fit models for each of the two data sets
# Pooled OLS: 2008–2012
pool_model_2008_2012 = PooledOLS.from_formula('ppg ~ age + I(age**2)', data=panel_2008_2012)
pool_result_2008_2012 = pool_model_2008_2012.fit()
print(pool_result_2008_2012.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                    ppg   R-squared:                        0.6687
Estimator:                  PooledOLS   R-squared (Between):              0.6214
No. Observations:                2266   R-squared (Within):               0.0534
Date:                Mon, Nov 02 2026   R-squared (Overall):              0.6687
Time:                        18:51:46   Log-likelihood                   -7219.6
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2284.4
Entities:                         743   P-value                           0.0000
Avg Obs:                       3.0498   Distribution:                  F(2,2264)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             2284.4
                            

In [26]:
#Pooled OLS: 2013–2017
pool_model_2013_2017 = PooledOLS.from_formula('ppg ~ age + I(age**2)', data=panel_2013_2017)
pool_result_2013_2017 = pool_model_2013_2017.fit()
print(pool_result_2013_2017.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                    ppg   R-squared:                        0.6757
Estimator:                  PooledOLS   R-squared (Between):              0.6103
No. Observations:                2403   R-squared (Within):               0.0318
Date:                Mon, Nov 02 2026   R-squared (Overall):              0.6757
Time:                        18:52:15   Log-likelihood                   -7587.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2501.2
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             2501.2
                            

In [27]:
#FE: 2008–2012
fe_model_2008_2012 = PanelOLS.from_formula('ppg ~ age + I(age**2) + EntityEffects', data=panel_2008_2012)
fe_result_2008_2012 = fe_model_2008_2012.fit()
print(fe_result_2008_2012.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    ppg   R-squared:                        0.2139
Estimator:                   PanelOLS   R-squared (Between):             -39.949
No. Observations:                2266   R-squared (Within):               0.2139
Date:                Mon, Nov 02 2026   R-squared (Overall):             -26.718
Time:                        18:52:25   Log-likelihood                   -4814.2
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      206.98
Entities:                         743   P-value                           0.0000
Avg Obs:                       3.0498   Distribution:                  F(2,1521)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             206.98
                            

In [28]:
# FE: 2013–2017
fe_model_2013_2017 = PanelOLS.from_formula('ppg ~ age + I(age**2) + EntityEffects', data=panel_2013_2017)
fe_result_2013_2017 = fe_model_2013_2017.fit()
print(fe_result_2013_2017.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    ppg   R-squared:                        0.1774
Estimator:                   PanelOLS   R-squared (Between):             -66.812
No. Observations:                2403   R-squared (Within):               0.1774
Date:                Mon, Nov 02 2026   R-squared (Overall):             -43.473
Time:                        18:53:38   Log-likelihood                   -5167.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      170.80
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,1584)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             170.80
                            

In [29]:
# RE: 2008–2012
re_model_2008_2012 = RandomEffects.from_formula('ppg ~ age + I(age**2)', data=panel_2008_2012)
re_result_2008_2012 = re_model_2008_2012.fit()
print(re_result_2008_2012.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                    ppg   R-squared:                        0.3868
Estimator:              RandomEffects   R-squared (Between):              0.6243
No. Observations:                2266   R-squared (Within):               0.1064
Date:                Mon, Nov 02 2026   R-squared (Overall):              0.6440
Time:                        18:53:42   Log-likelihood                   -5394.9
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      714.02
Entities:                         743   P-value                           0.0000
Avg Obs:                       3.0498   Distribution:                  F(2,2264)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             714.02
                            

In [30]:
# RE: 2013–2017
re_model_2013_2017 = RandomEffects.from_formula('ppg ~ age + I(age**2)', data=panel_2013_2017)
re_result_2013_2017 = re_model_2013_2017.fit()
print(re_result_2013_2017.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                    ppg   R-squared:                        0.3804
Estimator:              RandomEffects   R-squared (Between):              0.6371
No. Observations:                2403   R-squared (Within):               0.0412
Date:                Mon, Nov 02 2026   R-squared (Overall):              0.6573
Time:                        18:53:49   Log-likelihood                   -5810.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      737.02
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             737.02
                            

# (3) run diagnostics, compare, and interpret for 2008-2012.

In [31]:
# (3) run diagnostics, compare, and interpret for 2008-2012.
sg_2008_2012 = Stargazer([pool_result_2008_2012, fe_result_2008_2012, re_result_2008_2012])
sg_2008_2012.title('Points per Game and Age: 2008–2012')

# Label models and variables
sg_2008_2012.dependent_variable_name('Points per game')
sg_2008_2012.custom_columns(['Pooled OLS', 'Fixed Effects', 'Random Effects'], [1, 1, 1])
sg_2008_2012.covariate_order(['age', 'I(age ** 2)'])
sg_2008_2012.rename_covariates({
    'age': 'Age',
    'I(age ** 2)': 'Age-squared',
})

sg_2008_2012

In [32]:
# LM test
from scipy.stats import chi2

resid = pool_result_2008_2012.resids
e_i = resid.groupby(level=0).sum()
ssr = float((resid ** 2).sum())
A1 = float((e_i ** 2).sum()) / ssr - 1
T_i = resid.groupby(level=0).size()
n_obs = resid.shape[0]
lm_stat = (n_obs ** 2) / (2 * (float((T_i ** 2).sum()) - n_obs)) * A1 ** 2
lm_pval = chi2.sf(lm_stat, df=1)

print('LM test for individual effects: 2008–2012')
print(f'LM statistic = {lm_stat:.4f}')
print('df = 1')
print(f'p-value = {lm_pval:.4f}')

LM test for individual effects: 2008–2012
LM statistic = 2458.0017
df = 1
p-value = 0.0000


The null hypothesis is that there are no individual effects, meaning that there is no difference between players; the alternative is that there are individual effects. The p-value tells us there is a probability of < 0.001 that we obtain an estimated LM statistic of 2458.00 or more extreme if the true value of LM statistics = 0. It is less likely to happen under null, thus, we reject null. Hence, FE/RE is needed over pooled OLS as these model account for the unobserved, time-invariant variables. Hausman test is further needed to determine whether FE or RE is preferred.

In [33]:
# Hausman test
from scipy.stats import chi2

common_params = fe_result_2008_2012.params.index.intersection(re_result_2008_2012.params.index)

beta_diff = (fe_result_2008_2012.params[common_params] - re_result_2008_2012.params[common_params])
cov_diff = (
    fe_result_2008_2012.cov.loc[common_params, common_params] -
    re_result_2008_2012.cov.loc[common_params, common_params]
)

h_stat = float(beta_diff.T @ np.linalg.inv(cov_diff.values) @ beta_diff)
p_val = chi2.sf(h_stat, df=len(common_params))
print('Hausman test: 2008–2012')
print(f'Hausman statistic = {h_stat:.4f}')
print(f'df = {len(common_params)}')
print(f'p-value = {p_val:.4f}')
print('Prefer FE over RE:', p_val < 0.05)


Hausman test: 2008–2012
Hausman statistic = 271.5331
df = 2
p-value = 0.0000
Prefer FE over RE: True


Thus null hypothesis is that the unobserved time-invariant factors are uncorrelated with each explanatory variable; the laternative is that the unobserved time-invariant factors are correlated with each explanatory variable. The p-value tells us there is a probability of < 0.001 that we obtain an estimated Huasman statistic of 271.53 or more extreme if the true value of Hausman statistics = 0. It is less likely to happen under null, thus, we reject null. Hence, Fe is needed over RE as it account for the unobserved time-invariant factors.

# (4) run diagnostics, compare, and interpret for 2013-2017.

In [34]:
# (4) run diagnostics, compare, and interpret for 2013-2017.
sg_2013_2017 = Stargazer([pool_result_2013_2017, fe_result_2013_2017, re_result_2013_2017])
sg_2013_2017.title('Points per Game and Age: 2013–2017')

# Label models and variables
sg_2013_2017.dependent_variable_name('Points per game')
sg_2013_2017.custom_columns(['Pooled OLS', 'Fixed Effects', 'Random Effects'], [1, 1, 1])
sg_2013_2017.covariate_order(['age', 'I(age ** 2)'])
sg_2013_2017.rename_covariates({
    'age': 'Age',
    'I(age ** 2)': 'Age-squared',
})

sg_2013_2017

In [35]:
# LM test
from scipy.stats import chi2

resid = pool_result_2013_2017.resids
e_i = resid.groupby(level=0).sum()
ssr = float((resid ** 2).sum())
A1 = float((e_i ** 2).sum()) / ssr - 1
T_i = resid.groupby(level=0).size()
n_obs = resid.shape[0]
lm_stat = (n_obs ** 2) / (2 * (float((T_i ** 2).sum()) - n_obs)) * A1 ** 2
lm_pval = chi2.sf(lm_stat, df=1)

print('LM test for individual effects: 2013–2017')
print(f'LM statistic = {lm_stat:.4f}')
print('df = 1')
print(f'p-value = {lm_pval:.4f}')

LM test for individual effects: 2013–2017
LM statistic = 2381.2182
df = 1
p-value = 0.0000


The null hypothesis is that there are no individual effects, meaning that there is no difference between players; the alternative is that there are individual effects. The p-value tells us there is a probability of < 0.001 that we obtain an estimated LM statistic of 2381.22 or more extreme if the true value of LM statistics = 0. It is less likely to happen under null, thus, we reject null. Hence, FE/RE is needed over pooled OLS as these model account for the unobserved, time-invariant variables. Hausman test is further needed to determine whether FE or RE is preferred.

In [36]:
# Hausman test
import numpy as np
from scipy.stats import chi2

common_params = fe_result_2013_2017.params.index.intersection(re_result_2013_2017.params.index)

beta_diff = (fe_result_2013_2017.params[common_params] - re_result_2013_2017.params[common_params])
cov_diff = (
    fe_result_2013_2017.cov.loc[common_params, common_params] -
    re_result_2013_2017.cov.loc[common_params, common_params]
)

h_stat = float(beta_diff.T @ np.linalg.inv(cov_diff.values) @ beta_diff)
p_val = chi2.sf(h_stat, df=len(common_params))
print('Hausman test: 2013–2017')
print(f'Hausman statistic = {h_stat:.4f}')
print(f'df = {len(common_params)}')
print(f'p-value = {p_val:.4f}')
print('Prefer FE over RE:', p_val < 0.05)

Hausman test: 2013–2017
Hausman statistic = 285.0094
df = 2
p-value = 0.0000
Prefer FE over RE: True


Thus null hypothesis is that the unobserved time-invariant factors are uncorrelated with each explanatory variable; the laternative is that the unobserved time-invariant factors are correlated with each explanatory variable. The p-value tells us there is a probability of < 0.001 that we obtain an estimated Huasman statistic of 285.01 or more extreme if the true value of Hausman statistics = 0. It is less likely to happen under null, thus, we reject null. Hence, FE is needed over RE as it account for the unobserved time-invariant factors.

# (5) what differences do you find in the age-scoring relationship across the two eras?

In [ ]:
# (5) what differences do you find in the age-scoring relationship across the two eras?

Interpretation: According to the fixed effect models, both eras illustrate a quadratic relationship between age and points per game. This suggest that for both eras, initially, ppg increases as age increases. However, after reaching a turning point, ppg start to decreases as age decreases. The turning points are different for the two eras.